# HTCondor submission

This notebook illustrates how to generate parameters and training/test data using the HTCondor scheduler.

In [ ]:
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal
lal.swig_redirect_standard_output_error(False)

import os

from cogwheel_machine import compression, generate_parameters, simulation, utils

## 1. Set up run directory and configuration file
Let us make a run directory with a copy of the example `config.py` in it.

After this you can edit the new config file as needed.

In [ ]:
parentdir = '../data/'  # Edit as appropriate

rundir = utils.setup_rundir(parentdir)  # You may then edit the contents of the new config file

## 2. Generate simulation parameters

This will produce a dataframe `simulation_parameters.feather` with binary black hole parameters.

In [ ]:
accounting_group = 'ligo.dev.o4.cbc.pe.lalinference'  # Edit as needed

In [ ]:
generate_parameters.submit_condor(rundir,
                                  accounting_group=accounting_group, 
                                  accounting_group_user=os.environ['USER'])

Wait until it finishes

In [ ]:
!condor_q

## 3. Simulate signals and preprocess the data

This will produce directories for the training and test datasets, containing the following files:
* `preprocessed_data.npz`: Auxiliary file with heterodyned data, heterodyned signals, and phenomenological reference waveform parameters.
* `folded_sampled_params.npy`: Contains the true (injected) parameter values, after folding.
* `unfolding_labels.npy`: Contains the true (injected) index of the unfolding transformation. 

In [ ]:
simulation.submit_condor(rundir,
                         request_cpus=100,
                         request_memory='100G',
                         accounting_group=accounting_group, 
                         accounting_group_user=os.environ['USER'])

In [ ]:
!condor_q

## 4. Compress the data

This will create `compressed_data.npy`, which should be suitable for training the network.

In [ ]:
compression.create_mask(rundir)

In [ ]:
compression.submit_condor(rundir,
                          request_memory='32G',
                          accounting_group=accounting_group,
                          accounting_group_user=os.environ['USER'])

In [ ]:
!condor_q

## 5. Train the neural posterior estimator

First, set up a `modeldir` with its own `model_config` file:

In [ ]:
modeldir = utils.setup_modeldir(rundir)  # You may then edit the contents of the new config file

Again, edit the new config file as needed.

Then, preferably in a machine with a GPU, run:

    python -m cogwheel_machine.training {modeldir}

These last steps can be repeated as needed to train multiple models (e.g. varying the network architecture) without having to regenerate the data.